In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from utils import *
from interpolation import *
import pandas as pd
from show import *

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [2]:
df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/disk_centers.csv', skipinitialspace=True).drop(columns='date').dropna()
dids = df['did'].to_numpy()
xu_sun, yu_sun, ru_sun = df['xu_sun'].to_numpy(), df['yu_sun'].to_numpy(), df['ru_sun'].to_numpy()

s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion.npz')
xd, yd = s['xd'], s['yd']

In [3]:
files_fdt = sorted(glob.glob('/home/ulyanov/data/stereo/2025-09-24/fdt/*'))
files_fdt

['/home/ulyanov/data/stereo/2025-09-24/fdt/solo_L2_phi-fdt-bamb_20250924T061503_V202602220437_0549240502.fits.gz',
 '/home/ulyanov/data/stereo/2025-09-24/fdt/solo_L2_phi-fdt-bazi_20250924T061503_V202602220437_0549240502.fits.gz',
 '/home/ulyanov/data/stereo/2025-09-24/fdt/solo_L2_phi-fdt-binc_20250924T061503_V202602220437_0549240502.fits.gz',
 '/home/ulyanov/data/stereo/2025-09-24/fdt/solo_L2_phi-fdt-blos_20250924T061503_V202602220437_0549240502.fits.gz',
 '/home/ulyanov/data/stereo/2025-09-24/fdt/solo_L2_phi-fdt-vlos_20250924T061503_V202602220437_0549240502.fits.gz']

In [18]:
file_fdt = files_fdt[-1]

with fits.open(file_fdt) as hdul:
    header_fdt = hdul[0].header
    data_fdt = hdul[0].data


did = int(file_fdt.split('_')[-1].split('.')[0])
data_fdt = undistort(data_fdt, header_fdt, xd, yd) * 1000

view_fdt = View.from_header(header_fdt).update(xc=xu_sun[dids == did][0], yc=yu_sun[dids == did][0], rsun=ru_sun[dids == did][0],
                                               crota=header_fdt['CROTA'] + 0.25)

data_fdt -= view_fdt.velocity()

mu = view_fdt.mu()
t = np.where(mu > 0.1)
p = np.polyfit(mu[t], data_fdt[t], 3)
data_fdt -= np.polyval(p, mu)


view_new = view_fdt.update(nx=1024, ny=1024, rsun=480, xc=511.5, yc=511.5, crota=0, rsun_arc=0)
transform_fdt = (view_new.to_carrington() -
                 view_fdt.to_carrington(mu_thr=0.1))
grid, alpha = transform_fdt(view_new.grid())
data_fdt = bilinear(data_fdt, *grid) * alpha



show_data(data_fdt, view_new, 'FDT', cmap='seismic', vmin=-1000, vmax=1000)

In [5]:
files_hmi = sorted(glob.glob('/home/ulyanov/data/stereo/2025-09-24/hmi/*'))
files_hmi

['/home/ulyanov/data/stereo/2025-09-24/hmi/hmi.M_45s.20250924_062015_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/stereo/2025-09-24/hmi/hmi.V_45s.20250924_062015_TAI.2.Dopplergram.fits']

In [19]:
file_hmi = files_hmi[1]

with fits.open(file_hmi) as hdul:
    header_hmi = hdul[1].header
    data_hmi = hdul[1].data


data_hmi = rebin(data_hmi, 4, update_header=header_hmi)
view_hmi = View.from_header(header_hmi)

data_hmi -= view_hmi.velocity()
mu = view_hmi.mu()
t = np.where(mu > 0.1)
p = np.polyfit(mu[t], data_hmi[t], 3)
data_hmi -= np.polyval(p, mu)


transform_hmi = (view_new.to_carrington() -
                 view_hmi.to_carrington(mu_thr=0.1))

grid, alpha = transform_hmi(view_new.grid())
data_hmi = bilinear(data_hmi, *grid) * alpha

show_data(data_hmi, view_new, 'HMI', cmap='seismic', vmin=-1000, vmax=1000)

In [20]:
transform = view_hmi.to_carrington(origin='helioprojective') - view_fdt.to_carrington(origin='helioprojective')

e1 = (0,0,1)
e2 = transform(e1)[0]

e1 = np.array(e1)
e2 = np.array(e2)

q = np.sum(e1 * e2)


delta = np.sqrt(1 - q ** 2)

e2_ = (e2 - e1 * q) / delta

v1 = data_fdt.copy()
v2 = data_hmi.copy()

#v1[np.abs(v1) < 15] = np.nan
#v2[np.abs(v2) < 15] = np.nan

v2_ = (v2 - v1 * q) / delta


print(e2_, np.arccos(q) * 180 / np.pi)

[0.99926351 0.03837227 0.        ] 37.10844172814751


In [24]:
plt.figure(figsize=(10,10))
plt.imshow(v2_, 'seismic', vmin=-2000, vmax=2000)
plt.tight_layout()

In [16]:
plt.figure(figsize=(10,10))
plt.imshow(np.sqrt(v1 ** 2 + v2_ ** 2), 'jet', vmin=0, vmax=2000)
plt.tight_layout()

In [22]:
xi, yi, zi = view_new.grid(origin='helioprojective')

plt.figure(figsize=(10,10))
plt.imshow(np.abs(v1 * zi + v2_ * (e2_[0] * xi + e2_[1] * yi)) / np.sqrt(v1 ** 2 + v2_ ** 2), 'jet', vmin=0, vmax=1)
plt.tight_layout()

In [19]:
xi, yi, zi = view_new.grid(origin='helioprojective')

plt.figure(figsize=(10,10))
plt.imshow(v1 * zi / np.sqrt(v1 ** 2 + v2_ ** 2), 'jet', vmin=-1, vmax=1)
plt.tight_layout()